# Drivable-Area Segmentation to Action

This notebook shows how a segmentation-style computer vision output can be converted into a high-level driving action.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def make_scene(obstacle_center=False, lane_shift=0):
    h, w = 64, 64
    mask = np.zeros((h, w), dtype=int)
    center = w // 2 + lane_shift

    for y in range(h):
        half_width = int(8 + y * 0.22)
        left = max(0, center - half_width)
        right = min(w, center + half_width)
        mask[y, left:right] = 1

    if obstacle_center:
        mask[44:56, 28:36] = 2

    return mask

scenes = {
    "clear_lane": make_scene(False, 0),
    "obstacle_ahead": make_scene(True, 0),
    "lane_shift_left": make_scene(False, -10),
    "lane_shift_right": make_scene(False, 10),
}

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, (name, mask) in zip(axes, scenes.items()):
    ax.imshow(mask, cmap="viridis")
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()

In [ ]:
def extract_features(mask):
    h, w = mask.shape
    ego_window = mask[h-20:h-4, w//2-6:w//2+6]
    obstacle_ahead = np.any(ego_window == 2)

    road_pixels = np.argwhere(mask == 1)
    road_coverage = len(road_pixels) / mask.size
    lower_half = road_pixels[road_pixels[:, 0] > h // 2]

    if len(lower_half) == 0:
        lane_offset = 0
    else:
        lane_center = lower_half[:, 1].mean()
        lane_offset = lane_center - (w / 2)

    return {
        "road_coverage": road_coverage,
        "lane_offset": lane_offset,
        "obstacle_ahead": obstacle_ahead,
    }


def policy(features):
    if features["obstacle_ahead"]:
        return "stop", "Obstacle detected in the ego path."
    if features["road_coverage"] < 0.15:
        return "slow_down", "Low drivable-area confidence."
    if features["lane_offset"] < -5:
        return "steer_left", "Lane center is left of the ego centerline."
    if features["lane_offset"] > 5:
        return "steer_right", "Lane center is right of the ego centerline."
    return "keep_lane", "Drivable area is centered and clear."


for name, mask in scenes.items():
    features = extract_features(mask)
    action, reason = policy(features)
    print(name, features, "=>", action, "|", reason)

## Next Step

Swap the synthetic mask for a U-Net, DeepLab, SegFormer, or Mask2Former output trained on real driving scenes.